# Notebook guiado — K-Means desde cero

**RA2 · Programación de Inteligencia Artificial**  
**Notebook de refuerzo · Google Colab**

En este notebook aprenderemos **K-Means** de forma visual y progresiva.

La idea central es:

> **Agrupar observaciones parecidas sin conocer previamente a qué grupo pertenece cada una.**

K-Means es un algoritmo de **aprendizaje no supervisado** y, más concretamente, de **clustering**.

## Recorrido

1. Aprendizaje supervisado vs. no supervisado
2. Qué es clustering
3. Qué significa `k`
4. Centroides
5. Distancia y asignación
6. K-Means paso a paso
7. Inercia
8. Inicialización y `random_state`
9. Primer modelo con `scikit-learn`
10. Importancia del escalado
11. Método del codo
12. Silhouette Score
13. Visualización de silhouette por muestra
14. Interpretación de clusters
15. Segmentación de clientes ficticios
16. K-Means con más de dos variables
17. PCA para visualizar
18. Nuevas observaciones
19. Limitaciones de K-Means
20. Cuándo utilizarlo y cuándo no

> El objetivo no es memorizar código, sino entender **qué optimiza K-Means y por qué aparecen los clusters**.


# 1. Aprendizaje supervisado vs. no supervisado

En **aprendizaje supervisado** conocemos una variable objetivo.

Ejemplo:

| Edad | Ingresos | Compra |
|---:|---:|---|
| 24 | 22.000 | Sí |
| 46 | 48.000 | No |

El modelo aprende utilizando la respuesta correcta.

---

En **aprendizaje no supervisado** no disponemos de esa etiqueta objetivo.

Tenemos únicamente:

| Edad | Ingresos |
|---:|---:|
| 24 | 22.000 |
| 46 | 48.000 |
| 31 | 30.000 |

y queremos descubrir estructura en los datos.

Uno de los problemas más conocidos es el **clustering**:

> encontrar grupos de observaciones que sean internamente parecidas y diferentes de otros grupos.


# 2. ¿Qué es un cluster?

Un **cluster** es un grupo de observaciones que, según las variables utilizadas, presentan cierta similitud entre sí.

Ejemplos posibles:

- clientes con patrones de compra parecidos;
- productos con características similares;
- usuarios con comportamientos semejantes;
- documentos con representaciones próximas;
- sensores con patrones parecidos.

## Importante

K-Means **no sabe qué significa cada grupo**.

Puede producir:

- `cluster 0`
- `cluster 1`
- `cluster 2`

pero los nombres son simplemente identificadores.

La interpretación semántica la hacemos nosotros después analizando sus características.


# 3. ¿Qué significa la K de K-Means?

La `K` representa el **número de clusters que queremos encontrar**.

Si indicamos:

```python
KMeans(n_clusters=3)
```

estamos pidiendo al algoritmo que divida las observaciones en **3 grupos**.

Esto implica una diferencia importante respecto a algunos otros algoritmos de clustering:

> En K-Means debemos indicar `k` antes de ejecutar el algoritmo.

Por tanto, una pregunta clave será:

**¿Cómo elegimos un valor razonable de `k`?**

Más adelante utilizaremos:

- conocimiento del problema;
- método del codo;
- Silhouette Score;
- interpretación de los resultados.


# 4. Crear un ejemplo visual

Para entender el algoritmo utilizaremos datos sintéticos en dos dimensiones.

No les asignaremos ninguna etiqueta de clase al entrenar.

El gráfico mostrará simplemente las observaciones.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs

X_demo, _ = make_blobs(
    n_samples=120,
    centers=[
        [-4, -2],
        [0, 4],
        [5, 0]
    ],
    cluster_std=[1.0, 1.1, 1.2],
    random_state=42
)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    s=55
)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Datos sin etiquetas")
plt.grid(alpha=0.2)
plt.show()


A simple vista parece haber cierta estructura.

Pero el algoritmo no recibe el gráfico ni nuestra intuición.

Recibe solamente números:

\[
(x_1,x_2)
\]

y deberá construir los grupos utilizando distancias.


# 5. ¿Qué es un centroide?

Cada cluster de K-Means está representado por un **centroide**.

El centroide es el vector medio de las observaciones asignadas al cluster.

Si un cluster contiene:

\[
(1,2),\quad (3,4),\quad (5,3)
\]

su centroide será:

\[
\left(
\frac{1+3+5}{3},
\frac{2+4+3}{3}
\right)
=
(3,3)
\]

Por eso se llama **K-Means**:

> cada grupo se representa mediante su **media**.


In [ ]:
puntos = np.array([
    [1, 2],
    [3, 4],
    [5, 3]
])

centroide = puntos.mean(axis=0)

print("Puntos:")
print(puntos)

print("\nCentroide:")
print(centroide)


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    puntos[:, 0],
    puntos[:, 1],
    s=100
)
plt.scatter(
    centroide[0],
    centroide[1],
    marker="X",
    s=250,
    linewidths=2
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Puntos y su centroide")
plt.grid(alpha=0.2)
plt.show()


# 6. ¿Cómo funciona K-Means?

La versión conceptual del algoritmo puede resumirse en cuatro pasos:

### Paso 1 — Inicializar centroides

Elegimos `k` posiciones iniciales.

### Paso 2 — Asignar observaciones

Cada observación se asigna al centroide más cercano.

### Paso 3 — Recalcular centroides

Cada centroide se mueve hasta la media de las observaciones de su cluster.

### Paso 4 — Repetir

Volvemos a asignar observaciones y recalcular centroides.

El proceso continúa hasta alcanzar un criterio de convergencia: por ejemplo, cuando los centroides dejan de cambiar de forma relevante.


# 7. Vamos a construir algunas iteraciones manualmente

Utilizaremos:

\[
k=3
\]

y elegiremos tres centroides iniciales de forma didáctica.

Esto no pretende sustituir la implementación optimizada de `scikit-learn`.

Queremos **ver el movimiento de los centroides**.


In [ ]:
centroides = np.array([
    [-6.0, 4.0],
    [0.0, -3.0],
    [6.0, 5.0]
])

plt.figure(figsize=(8, 6))
plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    s=45,
    alpha=0.65
)
plt.scatter(
    centroides[:, 0],
    centroides[:, 1],
    marker="X",
    s=260,
    linewidths=2
)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Paso inicial: centroides antes de agrupar")
plt.grid(alpha=0.2)
plt.show()


## Paso de asignación

Para cada observación calculamos su distancia a cada centroide.

Después elegimos el centroide con menor distancia.


In [ ]:
def asignar_clusters(X, centroides):
    distancias = np.linalg.norm(
        X[:, np.newaxis, :] - centroides[np.newaxis, :, :],
        axis=2
    )

    return np.argmin(distancias, axis=1)


labels_iter_1 = asignar_clusters(
    X_demo,
    centroides
)

np.unique(
    labels_iter_1,
    return_counts=True
)


## Visualizar la primera asignación

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=labels_iter_1,
    s=50
)
plt.scatter(
    centroides[:, 0],
    centroides[:, 1],
    marker="X",
    s=260,
    linewidths=2
)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Iteración 1: asignación al centroide más próximo")
plt.grid(alpha=0.2)
plt.show()


## Paso de actualización

Ahora recalculamos cada centroide:

\[
\mu_j =
\frac{1}{|C_j|}
\sum_{\mathbf{x}\in C_j}\mathbf{x}
\]

Es decir:

> el nuevo centroide es la media de las observaciones que actualmente pertenecen a ese cluster.


In [ ]:
def recalcular_centroides(X, labels, k):
    nuevos = []

    for cluster in range(k):
        puntos_cluster = X[labels == cluster]
        nuevos.append(
            puntos_cluster.mean(axis=0)
        )

    return np.array(nuevos)


centroides_2 = recalcular_centroides(
    X_demo,
    labels_iter_1,
    k=3
)

print("Centroides anteriores:")
print(centroides)

print("\nNuevos centroides:")
print(centroides_2)


## Visualizar el movimiento de los centroides

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=labels_iter_1,
    s=45,
    alpha=0.6
)

plt.scatter(
    centroides[:, 0],
    centroides[:, 1],
    marker="X",
    s=210,
    linewidths=2,
    label="Centroides anteriores"
)

plt.scatter(
    centroides_2[:, 0],
    centroides_2[:, 1],
    marker="P",
    s=210,
    linewidths=2,
    label="Centroides nuevos"
)

for anterior, nuevo in zip(
    centroides,
    centroides_2
):
    plt.plot(
        [anterior[0], nuevo[0]],
        [anterior[1], nuevo[1]],
        linestyle="--"
    )

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Los centroides se desplazan hacia la media de su cluster")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Repetir varias iteraciones

Vamos a registrar cómo se mueven los centroides.


In [ ]:
def ejecutar_iteraciones_kmeans(
    X,
    centroides_iniciales,
    n_iter=6
):
    centroides = centroides_iniciales.copy()
    historial = [centroides.copy()]

    for _ in range(n_iter):
        labels = asignar_clusters(
            X,
            centroides
        )

        nuevos_centroides = recalcular_centroides(
            X,
            labels,
            k=len(centroides)
        )

        historial.append(
            nuevos_centroides.copy()
        )

        if np.allclose(
            nuevos_centroides,
            centroides
        ):
            break

        centroides = nuevos_centroides

    labels_finales = asignar_clusters(
        X,
        centroides
    )

    return centroides, labels_finales, historial


centroides_finales_manual, labels_manual, historial = (
    ejecutar_iteraciones_kmeans(
        X_demo,
        centroides,
        n_iter=10
    )
)

print(
    "Número de actualizaciones:",
    len(historial) - 1
)

print("\nCentroides finales:")
print(centroides_finales_manual)


## Visualizar la trayectoria completa

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=labels_manual,
    s=45,
    alpha=0.55
)

historial_array = np.array(historial)

for j in range(historial_array.shape[1]):
    plt.plot(
        historial_array[:, j, 0],
        historial_array[:, j, 1],
        marker="o",
        linestyle="--"
    )

plt.scatter(
    centroides_finales_manual[:, 0],
    centroides_finales_manual[:, 1],
    marker="X",
    s=280,
    linewidths=2
)

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Trayectoria de los centroides durante K-Means")
plt.grid(alpha=0.2)
plt.show()


Este gráfico resume la lógica del algoritmo:

**inicialización → asignación → nueva media → reasignación → nueva media → ...**

hasta estabilizarse.


# 8. ¿Qué intenta optimizar K-Means?

K-Means intenta formar clusters compactos respecto a sus centroides.

La función objetivo habitual es la suma de distancias cuadráticas dentro de los clusters:

\[
J =
\sum_{j=1}^{k}
\sum_{\mathbf{x}_i\in C_j}
\|\mathbf{x}_i-\boldsymbol{\mu}_j\|^2
\]

donde:

- \(C_j\) es el cluster \(j\);
- \(\boldsymbol{\mu}_j\) es su centroide.

En `scikit-learn`, el valor final de esta suma se expone mediante:

```python
model.inertia_
```

y suele llamarse **inercia**.


## Interpretación intuitiva de la inercia

- puntos muy próximos a sus centroides → menor inercia;
- puntos alejados de sus centroides → mayor inercia.

Pero cuidado:

> al aumentar `k`, la inercia tiende a disminuir.

Con suficientes clusters podríamos hacer grupos cada vez más pequeños.

Por eso **no debemos elegir `k` simplemente buscando la inercia mínima**.


# 9. K-Means con `scikit-learn`

Ahora utilizaremos la implementación real.

Indicaremos:

```python
n_clusters=3
```

y fijaremos `random_state` para que el ejemplo sea reproducible.


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init="auto"
)

labels = kmeans.fit_predict(
    X_demo
)

centros = kmeans.cluster_centers_

print("Inercia:", kmeans.inertia_)
print("\nCentroides:")
print(centros)


## Visualizar el resultado

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=labels,
    s=55
)

plt.scatter(
    centros[:, 0],
    centros[:, 1],
    marker="X",
    s=300,
    linewidths=2
)

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("K-Means con k=3")
plt.grid(alpha=0.2)
plt.show()


## `fit()`, `predict()` y `fit_predict()`

### `fit(X)`

Ajusta los centroides.

### `predict(X_new)`

Asigna nuevas observaciones al centroide más próximo.

### `fit_predict(X)`

Ajusta el modelo y devuelve directamente los clusters asignados a los datos utilizados.

En clustering es habitual utilizar `fit_predict()` durante la exploración.


# 10. Los números de cluster no tienen significado ordinal

Si K-Means produce:

- cluster 0;
- cluster 1;
- cluster 2;

eso **no significa**:

- 0 = peor;
- 1 = medio;
- 2 = mejor.

Tampoco existe una correspondencia fija entre el número y una categoría real.

Otra inicialización podría intercambiar los identificadores y producir la misma partición conceptual.

> Los IDs de cluster son etiquetas internas.


# 11. La inicialización importa

K-Means necesita posiciones iniciales para sus centroides.

Una mala inicialización puede conducir a una solución peor que otra.

Por eso las implementaciones modernas utilizan estrategias de inicialización como **k-means++** y pueden realizar varias inicializaciones.

En `scikit-learn` podemos controlar este comportamiento mediante parámetros como:

- `init`;
- `n_init`;
- `random_state`.

Para este curso no necesitamos memorizar la implementación de k-means++.

La idea importante es:

> K-Means puede converger a soluciones locales diferentes según la inicialización.


## Ejemplo con inicializaciones aleatorias

Ejecutaremos varias veces K-Means con `init="random"` y una única inicialización para ver que la solución puede variar.


In [ ]:
resultados_inicializacion = []

for seed in range(8):
    modelo = KMeans(
        n_clusters=3,
        init="random",
        n_init=1,
        random_state=seed
    )

    modelo.fit(X_demo)

    resultados_inicializacion.append({
        "seed": seed,
        "inertia": modelo.inertia_
    })

pd.DataFrame(
    resultados_inicializacion
).sort_values("inertia")


No debemos interpretar esto como que `random_state` sea un hiperparámetro que tengamos que optimizar.

Lo utilizamos principalmente para **reproducibilidad**.

El objetivo de varias inicializaciones es reducir la dependencia de una única selección inicial de centroides.


# 12. El escalado también es fundamental en K-Means

K-Means asigna puntos utilizando distancias.

Por tanto, aparece el mismo problema que vimos con KNN:

> una variable con números mucho mayores puede dominar la geometría del espacio.

Ejemplo de segmentación de clientes:

- visitas mensuales: 1–20;
- gasto anual: 1.000–60.000 €.

Sin escalado, la diferencia en euros puede dominar prácticamente toda la distancia.


## Crear un ejemplo exagerado

Partimos de dos variables con escalas muy diferentes.


In [ ]:
rng = np.random.default_rng(42)

n = 100

variable_pequena = np.concatenate([
    rng.normal(2, 0.5, n // 2),
    rng.normal(8, 0.6, n // 2)
])

variable_grande = np.concatenate([
    rng.normal(30000, 5000, n // 2),
    rng.normal(35000, 5000, n // 2)
])

X_escalas = np.column_stack([
    variable_pequena,
    variable_grande
])

df_escalas = pd.DataFrame(
    X_escalas,
    columns=[
        "visitas_mes",
        "gasto_anual"
    ]
)

df_escalas.head()


## K-Means sin escalar


In [ ]:
km_sin_escalar = KMeans(
    n_clusters=2,
    random_state=42,
    n_init="auto"
)

labels_sin = km_sin_escalar.fit_predict(
    X_escalas
)

plt.figure(figsize=(8, 6))
plt.scatter(
    df_escalas["visitas_mes"],
    df_escalas["gasto_anual"],
    c=labels_sin,
    s=55
)
plt.xlabel("Visitas al mes")
plt.ylabel("Gasto anual (€)")
plt.title("K-Means sin escalado")
plt.grid(alpha=0.2)
plt.show()


## Estandarizar antes de agrupar

`StandardScaler` transforma cada característica utilizando:

\[
z=\frac{x-\mu}{\sigma}
\]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_escalado = scaler.fit_transform(
    X_escalas
)

km_escalado = KMeans(
    n_clusters=2,
    random_state=42,
    n_init="auto"
)

labels_escalados = km_escalado.fit_predict(
    X_escalado
)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_escalado[:, 0],
    X_escalado[:, 1],
    c=labels_escalados,
    s=55
)
plt.xlabel("Visitas al mes — estandarizadas")
plt.ylabel("Gasto anual — estandarizado")
plt.title("K-Means después de estandarizar")
plt.grid(alpha=0.2)
plt.show()


## Mensaje importante

El escalado cambia la geometría y, por tanto, puede cambiar los clusters.

No debemos escalar de forma automática sin pensar.

Debemos preguntarnos:

> ¿Qué significado queremos que tenga la distancia entre observaciones?

En muchos problemas con variables numéricas en escalas diferentes, estandarizar es una decisión razonable.


# 13. ¿Cómo elegir `k`? Método del codo

Una estrategia exploratoria es entrenar modelos con distintos valores de `k` y observar la **inercia**.

La inercia siempre tenderá a reducirse al aumentar `k`.

Buscamos una zona donde añadir más clusters produzca una mejora cada vez menor.

A esa forma se la conoce como **codo**.

> El método del codo es una heurística visual, no una regla matemática infalible.


In [ ]:
valores_k = range(1, 11)
inercias = []

for k in valores_k:
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    modelo.fit(X_demo)

    inercias.append(
        modelo.inertia_
    )

tabla_codo = pd.DataFrame({
    "k": list(valores_k),
    "inercia": inercias
})

tabla_codo


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(
    tabla_codo["k"],
    tabla_codo["inercia"],
    marker="o"
)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.grid(alpha=0.2)
plt.show()


En estos datos sintéticos esperamos que una solución alrededor de tres clusters resulte razonable.

Pero no debemos elegir `k` únicamente por este gráfico.

Conviene combinar varias fuentes de evidencia.


# 14. Silhouette Score

Otra herramienta es el **Silhouette Score**.

Para cada observación compara:

- \(a\): distancia media a puntos de su propio cluster;
- \(b\): distancia media al cluster alternativo más próximo.

El coeficiente de silhouette de una observación es:

\[
s =
\frac{b-a}{\max(a,b)}
\]

Su rango está entre:

\[
-1 \leq s \leq 1
\]

Interpretación aproximada:

- próximo a 1 → observación bien integrada en su cluster;
- próximo a 0 → observación cerca de una frontera;
- negativo → podría estar más cerca de otro cluster.

El `silhouette_score` resume el valor medio de todas las observaciones.


In [ ]:
from sklearn.metrics import silhouette_score

resultados_silhouette = []

for k in range(2, 9):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    labels_k = modelo.fit_predict(
        X_demo
    )

    score = silhouette_score(
        X_demo,
        labels_k
    )

    resultados_silhouette.append({
        "k": k,
        "silhouette": score
    })

tabla_silhouette = pd.DataFrame(
    resultados_silhouette
)

tabla_silhouette


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(
    tabla_silhouette["k"],
    tabla_silhouette["silhouette"],
    marker="o"
)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score medio")
plt.title("Silhouette Score según k")
plt.grid(alpha=0.2)
plt.show()


## No existe una única métrica mágica

No debemos pensar:

> “el `k` con mayor silhouette es automáticamente el correcto”.

También importan:

- utilidad del problema;
- estabilidad;
- interpretabilidad;
- tamaño de los clusters;
- forma de los datos;
- conocimiento del dominio.

Las métricas ayudan a tomar decisiones, pero no sustituyen el análisis.


# 15. Ver el silhouette de cada observación

El promedio puede ocultar información.

Vamos a calcular el coeficiente de silhouette individual para cada punto con `k=3`.

Así podremos ver qué observaciones están:

- claramente dentro de un cluster;
- cerca de una frontera;
- potencialmente mal asignadas.


In [ ]:
from sklearn.metrics import silhouette_samples

modelo_sil = KMeans(
    n_clusters=3,
    random_state=42,
    n_init="auto"
)

labels_sil = modelo_sil.fit_predict(
    X_demo
)

sil_samples = silhouette_samples(
    X_demo,
    labels_sil
)

df_sil = pd.DataFrame({
    "cluster": labels_sil,
    "silhouette": sil_samples
})

df_sil.groupby("cluster")["silhouette"].agg(
    ["mean", "min", "max", "count"]
)


## Gráfico de silhouette por cluster

Cada línea horizontal representa una observación.


In [ ]:
plt.figure(figsize=(9, 6))

y_lower = 10

for cluster in sorted(
    np.unique(labels_sil)
):
    valores = np.sort(
        sil_samples[
            labels_sil == cluster
        ]
    )

    tamaño = len(valores)
    y_upper = y_lower + tamaño

    plt.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        valores,
        alpha=0.7
    )

    plt.text(
        -0.05,
        y_lower + tamaño / 2,
        str(cluster)
    )

    y_lower = y_upper + 10

plt.axvline(
    silhouette_score(
        X_demo,
        labels_sil
    ),
    linestyle="--",
    label="Media"
)

plt.xlabel("Silhouette Coefficient")
plt.ylabel("Observaciones agrupadas por cluster")
plt.title("Silhouette por observación — k=3")
plt.legend()
plt.show()


# 16. Un cluster no sirve hasta que lo interpretamos

K-Means devuelve números:

- 0;
- 1;
- 2.

Eso no es todavía conocimiento útil.

Después debemos analizar:

- medias;
- medianas;
- tamaños;
- distribuciones;
- variables características de cada cluster.

Por ejemplo, en segmentación de clientes podríamos descubrir:

- cluster A → visitas frecuentes y gasto alto;
- cluster B → visitas ocasionales y gasto bajo;
- cluster C → pocas visitas pero compras grandes.

Los nombres aparecen **después del análisis**, no los descubre K-Means por sí solo.


# 17. Caso práctico — Segmentación de clientes ficticios

Vamos a construir un dataset con cuatro variables:

- `edad`;
- `visitas_mes`;
- `gasto_medio`;
- `compras_online_pct`.

No tendremos una columna llamada “tipo de cliente”.

El objetivo será descubrir perfiles mediante clustering.


In [ ]:
rng = np.random.default_rng(123)

n_por_grupo = 70

clientes_1 = pd.DataFrame({
    "edad": rng.normal(24, 4, n_por_grupo),
    "visitas_mes": rng.normal(11, 2, n_por_grupo),
    "gasto_medio": rng.normal(35, 7, n_por_grupo),
    "compras_online_pct": rng.normal(80, 8, n_por_grupo)
})

clientes_2 = pd.DataFrame({
    "edad": rng.normal(45, 6, n_por_grupo),
    "visitas_mes": rng.normal(4, 1.2, n_por_grupo),
    "gasto_medio": rng.normal(90, 15, n_por_grupo),
    "compras_online_pct": rng.normal(35, 10, n_por_grupo)
})

clientes_3 = pd.DataFrame({
    "edad": rng.normal(34, 5, n_por_grupo),
    "visitas_mes": rng.normal(7, 1.5, n_por_grupo),
    "gasto_medio": rng.normal(55, 10, n_por_grupo),
    "compras_online_pct": rng.normal(55, 10, n_por_grupo)
})

clientes = pd.concat(
    [clientes_1, clientes_2, clientes_3],
    ignore_index=True
)

clientes["visitas_mes"] = clientes[
    "visitas_mes"
].clip(lower=0)

clientes["gasto_medio"] = clientes[
    "gasto_medio"
].clip(lower=0)

clientes["compras_online_pct"] = clientes[
    "compras_online_pct"
].clip(0, 100)

clientes.head()


## Explorar las variables

In [ ]:
clientes.describe().round(2)


## Visualización parcial de dos variables

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    clientes["visitas_mes"],
    clientes["gasto_medio"],
    s=50,
    alpha=0.65
)
plt.xlabel("Visitas al mes")
plt.ylabel("Gasto medio (€)")
plt.title("Clientes antes de aplicar K-Means")
plt.grid(alpha=0.2)
plt.show()


## Escalar las cuatro variables y entrenar

Ahora K-Means utilizará **las cuatro características**, no solamente las dos visibles en el gráfico.


In [ ]:
from sklearn.pipeline import Pipeline

pipeline_clientes = Pipeline([
    ("scaler", StandardScaler()),
    (
        "kmeans",
        KMeans(
            n_clusters=3,
            random_state=42,
            n_init="auto"
        )
    )
])

clusters_clientes = pipeline_clientes.fit_predict(
    clientes
)

clientes_cluster = clientes.copy()
clientes_cluster["cluster"] = clusters_clientes

clientes_cluster.head()


## Tamaño de cada cluster

In [ ]:
clientes_cluster["cluster"].value_counts().sort_index()


# 18. Perfilar los clusters

Calcularemos las medias de las variables originales para interpretar cada grupo.


In [ ]:
perfil_clusters = (
    clientes_cluster
    .groupby("cluster")
    .mean()
    .round(2)
)

perfil_clusters


Esta tabla es mucho más útil que limitarse a mirar los IDs.

Ahora podemos intentar describir los perfiles.

### Preguntas

- ¿Qué cluster tiene mayor gasto medio?
- ¿Cuál visita más frecuentemente?
- ¿Cuál utiliza más el canal online?
- ¿Qué diferencias de edad aparecen?
- ¿Qué nombre empresarial pondrías a cada segmento?


## Recuperar los centroides en las unidades originales

Como entrenamos sobre datos estandarizados, los centroides internos también están en espacio estandarizado.

Podemos aplicar la transformación inversa para volver a:

- años;
- visitas;
- euros;
- porcentaje.


In [ ]:
scaler_clientes = pipeline_clientes.named_steps[
    "scaler"
]

kmeans_clientes = pipeline_clientes.named_steps[
    "kmeans"
]

centros_originales = scaler_clientes.inverse_transform(
    kmeans_clientes.cluster_centers_
)

df_centros = pd.DataFrame(
    centros_originales,
    columns=clientes.columns
)

df_centros.index.name = "cluster"

df_centros.round(2)


En K-Means, el centroide funciona como un **perfil medio representativo** del cluster.

Eso no significa que exista necesariamente un cliente real exactamente igual al centroide.


## Visualizar los clusters en dos variables

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    clientes_cluster["visitas_mes"],
    clientes_cluster["gasto_medio"],
    c=clientes_cluster["cluster"],
    s=55,
    alpha=0.7
)

plt.scatter(
    df_centros["visitas_mes"],
    df_centros["gasto_medio"],
    marker="X",
    s=300,
    linewidths=2
)

plt.xlabel("Visitas al mes")
plt.ylabel("Gasto medio (€)")
plt.title("Segmentación de clientes con K-Means")
plt.grid(alpha=0.2)
plt.show()


Recuerda:

> El modelo utilizó cuatro variables, aunque este gráfico solo muestra dos.

Un gráfico 2D es una **proyección parcial** del resultado.


# 19. ¿Cómo visualizar clustering con muchas variables?

Con 4, 20 o 100 características no podemos dibujar directamente el espacio completo.

Una posibilidad es utilizar **PCA** para obtener una representación en dos dimensiones.

PCA:

- crea nuevas variables llamadas componentes principales;
- intenta conservar tanta variabilidad como sea posible;
- permite representar los datos en menos dimensiones.

## Importante

Aquí utilizaremos PCA **para visualizar**.

Los clusters ya han sido calculados previamente usando las cuatro variables estandarizadas.


In [ ]:
from sklearn.decomposition import PCA

X_clientes_escalado = scaler_clientes.transform(
    clientes
)

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_clientes_escalado
)

print(
    "Varianza explicada por PC1 + PC2:",
    round(
        pca.explained_variance_ratio_.sum(),
        3
    )
)


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters_clientes,
    s=55,
    alpha=0.7
)

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clusters representados mediante PCA")
plt.grid(alpha=0.2)
plt.show()


PCA no convierte mágicamente un problema complejo en una imagen perfecta.

Es una representación reducida.

Siempre debemos recordar cuánto de la información original estamos conservando.


# 20. Asignar un cliente nuevo

Una vez entrenado K-Means, podemos asignar nuevas observaciones al cluster cuyo centroide quede más próximo.

Ejemplo:

- edad: 27;
- 10 visitas al mes;
- gasto medio: 40 €;
- 78 % de compras online.


In [ ]:
cliente_nuevo = pd.DataFrame({
    "edad": [27],
    "visitas_mes": [10],
    "gasto_medio": [40],
    "compras_online_pct": [78]
})

cluster_nuevo = pipeline_clientes.predict(
    cliente_nuevo
)[0]

print(
    "Cluster asignado:",
    cluster_nuevo
)

print("\nPerfil medio del cluster:")
display(
    perfil_clusters.loc[
        [[cluster_nuevo]]
    ]
)


Esto no convierte K-Means en un algoritmo supervisado.

Seguimos sin tener una etiqueta objetivo.

Simplemente asignamos el nuevo punto al cluster aprendido que le corresponde según la geometría del modelo.


# 21. ¿Y si no supiéramos que queremos 3 segmentos?

En un caso real no tendríamos por qué conocer `k`.

Podemos explorar diferentes opciones mediante codo y silhouette sobre los clientes estandarizados.


In [ ]:
comparacion_k = []

for k in range(2, 8):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    labels_tmp = modelo.fit_predict(
        X_clientes_escalado
    )

    comparacion_k.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(
            X_clientes_escalado,
            labels_tmp
        )
    })

comparacion_k = pd.DataFrame(
    comparacion_k
)

comparacion_k


## Gráfico de inercia

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    comparacion_k["k"],
    comparacion_k["inercia"],
    marker="o"
)
plt.xlabel("k")
plt.ylabel("Inercia")
plt.title("Método del codo — clientes")
plt.grid(alpha=0.2)
plt.show()


## Gráfico de silhouette

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    comparacion_k["k"],
    comparacion_k["silhouette"],
    marker="o"
)
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette según k — clientes")
plt.grid(alpha=0.2)
plt.show()


La decisión final debería combinar:

- métricas;
- visualización;
- estabilidad;
- tamaños de los grupos;
- utilidad empresarial;
- interpretabilidad.

Por ejemplo, seis clusters pueden mejorar una métrica pero ser inútiles si el negocio necesita tres segmentos claramente accionables.


# 22. Limitación: K-Means no sirve igual de bien para cualquier geometría

K-Means representa cada cluster mediante un centroide y utiliza distancia euclídea.

Por eso funciona especialmente bien cuando los grupos tienen una geometría relativamente compacta.

Puede tener dificultades cuando:

- los clusters tienen formas no convexas;
- existen densidades muy diferentes;
- existen tamaños muy distintos;
- hay muchos outliers;
- la media no representa bien al grupo.

Vamos a observar un ejemplo clásico.


In [ ]:
from sklearn.datasets import make_moons

X_moons, _ = make_moons(
    n_samples=300,
    noise=0.07,
    random_state=42
)

km_moons = KMeans(
    n_clusters=2,
    random_state=42,
    n_init="auto"
)

labels_moons = km_moons.fit_predict(
    X_moons
)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_moons[:, 0],
    X_moons[:, 1],
    c=labels_moons,
    s=50
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("K-Means sobre clusters con forma no convexa")
plt.grid(alpha=0.2)
plt.show()


El algoritmo debe crear dos regiones organizadas alrededor de centroides.

Eso no coincide necesariamente con las dos “lunas” que intuitivamente vemos.

Otros algoritmos, como **DBSCAN**, siguen una lógica diferente y pueden adaptarse mejor a algunas geometrías.

> No existe un algoritmo de clustering universalmente mejor.


# 23. Sensibilidad a outliers

El centroide utiliza una **media**.

Las medias pueden desplazarse por observaciones extremas.

Veamos qué ocurre si añadimos un punto muy alejado.


In [ ]:
X_outlier = np.vstack([
    X_demo,
    [18, 18]
])

km_outlier = KMeans(
    n_clusters=3,
    random_state=42,
    n_init="auto"
)

labels_outlier = km_outlier.fit_predict(
    X_outlier
)

centros_outlier = km_outlier.cluster_centers_

plt.figure(figsize=(8, 6))

plt.scatter(
    X_outlier[:, 0],
    X_outlier[:, 1],
    c=labels_outlier,
    s=50
)

plt.scatter(
    centros_outlier[:, 0],
    centros_outlier[:, 1],
    marker="X",
    s=280,
    linewidths=2
)

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Efecto de una observación extrema")
plt.grid(alpha=0.2)
plt.show()


# 24. Pipeline mental de K-Means

Cuando trabajes con K-Means:

### 1. Elegir variables

¿Qué significa “parecido” en este problema?

### 2. Limpiar los datos

Valores ausentes, errores y observaciones extremas pueden afectar al resultado.

### 3. Revisar escalas

Si las variables están en unidades diferentes, considerar escalado.

### 4. Explorar posibles valores de `k`

Utilizar:

- conocimiento del dominio;
- codo;
- silhouette;
- interpretación.

### 5. Ajustar K-Means

```python
KMeans(...)
```

### 6. Analizar tamaños y centroides

Los IDs por sí solos no significan nada.

### 7. Interpretar los clusters

Convertir:

`cluster 0`

en algo útil como:

`clientes frecuentes de gasto moderado`

solo si los datos realmente apoyan esa interpretación.

### 8. Validar su utilidad

Un clustering matemáticamente razonable puede no ser útil para el problema real.


# 25. Errores frecuentes

## 1. Pensar que cluster 0, 1 y 2 son categorías conocidas

No. K-Means descubre una partición sin conocer etiquetas.

---

## 2. Comparar directamente IDs de clusters entre dos ejecuciones

Los identificadores pueden intercambiarse.

---

## 3. No escalar variables con unidades muy diferentes

Puede alterar completamente la geometría.

---

## 4. Elegir `k` únicamente por el codo

Es una heurística, no una respuesta automática.

---

## 5. Elegir `k` únicamente por silhouette

Una métrica no sustituye la utilidad e interpretación del problema.

---

## 6. Suponer que K-Means encontrará cualquier forma de grupo

Su geometría basada en centroides tiene limitaciones.

---

## 7. Interpretar clusters sin estudiar sus variables

El algoritmo devuelve números; el significado requiere análisis.

---

## 8. Utilizar variables sin sentido solo porque están disponibles

Cada variable modifica las distancias y puede cambiar el resultado.


# 26. Mini reto final

Utiliza el dataset `clientes` de este notebook.

Sin copiar directamente las celdas anteriores:

1. explora las variables;
2. estandarízalas;
3. calcula K-Means para `k = 2, 3, 4, 5 y 6`;
4. muestra la curva del codo;
5. calcula Silhouette Score;
6. elige un `k` y justifica la decisión;
7. entrena el modelo final;
8. añade la columna `cluster`;
9. calcula el perfil medio de cada grupo;
10. asigna un nombre descriptivo a cada cluster;
11. crea al menos un gráfico útil;
12. explica una limitación de tu segmentación.


In [ ]:
# TU CÓDIGO AQUÍ



# 27. Preguntas de comprensión

1. ¿Por qué K-Means es no supervisado?
2. ¿Qué representa `k`?
3. ¿Qué es un centroide?
4. ¿Cómo se asigna un punto a un cluster?
5. ¿Cómo se recalcula un centroide?
6. ¿Cuándo termina K-Means?
7. ¿Qué representa la inercia?
8. ¿Por qué la inercia disminuye normalmente al aumentar `k`?
9. ¿Qué intenta mostrar el método del codo?
10. ¿Qué representa un Silhouette Score próximo a 1?
11. ¿Por qué es importante escalar algunas variables?
12. ¿Qué función tiene `random_state`?
13. ¿Por qué las etiquetas 0, 1 y 2 no tienen significado semántico?
14. ¿Cómo interpretarías un cluster después de entrenar?
15. ¿Por qué K-Means puede fallar con formas como dos lunas?
16. ¿Cómo pueden afectar los outliers?
17. ¿Para qué puede utilizarse PCA después del clustering?
18. ¿PCA participa necesariamente en el entrenamiento de K-Means en nuestro ejemplo?
19. ¿Podemos asignar una observación nueva a un cluster?
20. ¿Por qué un clustering con buena métrica puede no ser útil para un negocio?


# 28. Qué debes recordar

K-Means intenta construir grupos alrededor de centroides.

El proceso fundamental es:

**elegir k centroides**

↓

**asignar cada punto al centroide más cercano**

↓

**recalcular cada centroide como la media**

↓

**repetir hasta converger**

Para utilizarlo correctamente debemos recordar:

- es un algoritmo no supervisado;
- `k` debe elegirse;
- los números de cluster no tienen significado por sí solos;
- las distancias hacen que el escalado sea importante;
- la inicialización puede afectar al resultado;
- la inercia mide compactación respecto a centroides;
- silhouette aporta otra perspectiva sobre cohesión y separación;
- los clusters deben interpretarse;
- K-Means tiene supuestos geométricos y limitaciones;
- no existe un `k` ni un algoritmo de clustering universalmente mejor.

## Idea final

> **K-Means no “descubre la verdad escondida” de un dataset. Construye una partición matemática según las variables, escalas, distancia y número de clusters que nosotros hemos elegido.**

La parte importante empieza después:

> **¿Tiene esa partición sentido y utilidad para el problema real?**
